# Notebook 05 (Oscar): MLP Training -- Exp 3 and Exp 4

**Exp 3:** Delta Residue + Full Wildtype -- `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Exp 4:** Delta Sequence -- `mean_pool(mutant_seq) - mean_pool(wt_seq)` (cached)

Both experiments run for both ESM-2 and AbLang2. Training uses MSE loss.
Evaluation metric: Spearman correlation per dataset and aggregate (excluding HER2 separately).
All runs logged to W&B.

## Setup

In [11]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Environment: local
Repo: /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project


Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [12]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

Drive root:      /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project
Embedding dir:   /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/embeddings
Checkpoint dir:  /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/checkpoints
Paths set.


`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [13]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

Local run -- installation skipped.


In [14]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Autoreload enabled.


In [15]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device: mps
Apple MPS -- Apple Silicon unified memory


Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [16]:
import numpy as np
import pandas as pd
import wandb
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

Imports OK.


## Data Loading and Splits

In [17]:
df = load_abagym_antibody(DATA_DIR)

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")
print()

# Verify all datasets are represented in each split
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    counts = df.iloc[idx]['DMS_name'].value_counts().to_dict()
    print(f"{split_name}: {counts}")

Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}


Loads the AbAgym metadata CSV and creates the stratified 80/10/10 split.

The split is stratified within each antibody dataset separately, then pooled.
This ensures all 5 antibodies are represented in train, val, and test.
The `random_state=42` is fixed -- Lucas uses the same value in NB05_lucas.ipynb
to guarantee identical splits across both notebooks.

Confirmed output:

```
Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val:   {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test:  {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
```

All 5 antibodies present in every split. HER2 val/test N=18 -- Spearman on 18 samples
is noisy; report but note unreliability. Splits are identical to Lucas's notebook.

## Experiment 4: Delta Sequence

**Input:** `mean_pool(mutant_seq) - mean_pool(wt_seq)`  
**Dims:** ESM-2 = 2560, AbLang2 = 960  
**Owner:** Oscar

The simplest and most direct embedding strategy. The delta sequence vector captures
the antibody-wide shift in embedding space caused by the mutation. From NB04 EDA,
the L2 norm of this vector is already weakly predictive of mutation effect
(Spearman r=0.083 ESM-2, r=0.216 AbLang2) without any supervised training.
A trained MLP operating on the full 2560/960-dim vector has access to directional
information that the norm discards, so performance should improve substantially.

In [20]:
# Build datasets for both models -- Exp 4 (DELTA_SEQUENCE)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_SEQUENCE,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_SEQUENCE: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

esm2 DELTA_SEQUENCE: input_dim=2560, label=0.7380, region=CDR_H3
ablang2 DELTA_SEQUENCE: input_dim=960, label=0.7380, region=CDR_H3


Sanity check: verifies that the dataset loads correctly and the input dimension
matches expectations (ESM-2=2560, AbLang2=960).

Confirmed output:

```
esm2 DELTA_SEQUENCE: input_dim=2560, label=0.7380, region=CDR_H3
ablang2 DELTA_SEQUENCE: input_dim=960, label=0.7380, region=CDR_H3
```

Input dims correct. Both models return the same label and region for row 0
(Ang2_2017_G6 H:P100A, CDR_H3, MinMax score=0.7380).

## Experiment 3: Delta Residue + Full Wildtype

**Input:** `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Dims:** ESM-2 = 1280 + 2560 = 3840, AbLang2 = 480 + 960 = 1440  
**Owner:** Oscar

Combines the local mutation signal (per-token delta at the mutation site) with
global antibody context (wildtype sequence embedding). The rationale: the per-token
delta alone tells you how much the mutation changed that position, but the wildtype
context tells the MLP what kind of antibody this is (CDR vs FR context, scaffold
type, chain identity). Together they provide both local and global information.

The wildtype embedding is shared across all mutations of the same antibody --
the dataset class handles the index lookup internally.

In [22]:
# Build datasets for both models -- Exp 3 (DELTA_RESIDUE_PLUS_WILD)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_RESIDUE_PLUS_WILD: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

esm2 DELTA_RESIDUE_PLUS_WILD: input_dim=3840, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE_PLUS_WILD: input_dim=1440, label=0.7380, region=CDR_H3


Sanity check: verifies input dimensions for Exp 3
(ESM-2=3840, AbLang2=1440) and that the wildtype index lookup works.

Confirmed output:

```
esm2 DELTA_RESIDUE_PLUS_WILD: input_dim=3840, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE_PLUS_WILD: input_dim=1440, label=0.7380, region=CDR_H3
```

Input dims correct (1280 delta_residue + 2560 wt_sequence for ESM-2;
480 + 960 for AbLang2). Wildtype index inversion verified -- DMS_name lookup
returns the correct wt_sequence row.

## Experiment 4: Training

Run for both ESM-2 and AbLang2. Each call to `train_abagym` logs a separate
W&B run. Results are stored in `exp4_results` for downstream test evaluation.

**Expected input dims:** ESM-2 = 2560, AbLang2 = 960  
**Architecture:** [256, 128] hidden layers, ReLU + Dropout(0.1)  
**Early stopping:** patience = 10 epochs on aggregate val Spearman

In [ ]:
exp4_results = {}

for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_SEQUENCE,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    # Infer input_dim from one sample
    x0, _, _ = ds_full[0]
    input_dim = x0.shape[0]
    print(f"{model_name} DELTA_SEQUENCE -- input_dim={input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy='delta_sequence',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    exp4_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print(f"  Per-dataset: {result['per_dataset_spearman']}")
    print()

Record confirmed best epoch, val Spearman (all 5 datasets), and per-dataset
breakdown here after running. Note whether HER2 performance differs substantially
from the other four antibodies.

Expected range for aggregate Spearman: 0.1–0.4 (rough prior from EDA norm baseline).

## Experiment 4: Test Evaluation

Evaluate the best-epoch model (restored by `train_abagym`) on the held-out
test split. Report per-dataset Spearman, aggregate (all 5), and aggregate
excluding HER2.

In [ ]:
print("=== Experiment 4: DELTA_SEQUENCE -- Test Results ===")
for model_name, result in exp4_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()}")
    print(f"  Aggregate Spearman (all 5):   {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

Record confirmed test Spearman results here. Compare ESM-2 vs AbLang2 performance
across datasets. Flag any dataset where one model substantially outperforms the other.
Note HER2 results separately -- its bimodal distribution makes Spearman less stable.

## Experiment 3: Training

Run for both ESM-2 and AbLang2. Same architecture and hyperparameters as Exp 4.

**Expected input dims:** ESM-2 = 3840, AbLang2 = 1440  
**Architecture:** [256, 128] hidden layers, ReLU + Dropout(0.1)

The larger input from Exp 3 means the first projection layer (3840→256 or
1440→256) compresses more aggressively. If Exp 3 doesn't clearly improve over
Exp 4, the wildtype context is not adding useful signal beyond what the delta
already encodes.

In [ ]:
exp3_results = {}

for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    x0, _, _ = ds_full[0]
    input_dim = x0.shape[0]
    print(f"{model_name} DELTA_RESIDUE_PLUS_WILD -- input_dim={input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy='delta_residue_plus_wild',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    exp3_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print(f"  Per-dataset: {result['per_dataset_spearman']}")
    print()

Record confirmed results here. Key question: does adding wildtype context improve
over Exp 4 (delta sequence only)? Compare directly after running both experiments.

## Experiment 3: Test Evaluation

In [ ]:
print("=== Experiment 3: DELTA_RESIDUE_PLUS_WILD -- Test Results ===")
for model_name, result in exp3_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()}")
    print(f"  Aggregate Spearman (all 5):   {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

Record test results here. Compare to Exp 4 test numbers directly.
If Exp 3 ≤ Exp 4, the wildtype context is redundant given the delta sequence.

## Summary: Exp 3 vs Exp 4

In [ ]:
rows = []
for exp_name, results_dict in [('Exp4_DeltaSeq', exp4_results), ('Exp3_DeltaRes+WT', exp3_results)]:
    for model_name, result in results_dict.items():
        metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
        row = {
            'Experiment': exp_name,
            'Model': model_name,
            'Spearman_all': round(metrics['aggregate'], 4),
            'Spearman_excl_HER2': round(metrics['exclude_her2'], 4),
            'HER2': round(metrics['HER2'], 4),
        }
        for ds, r in metrics['per_dataset'].items():
            row[ds] = round(r, 4)
        rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

Record the full summary table here after running. This table is the primary
deliverable for Oscar's Exp 3 and Exp 4 experiments and feeds into the
cross-experiment comparison in the final results notebook.